# Lab 02 — Spark SQL, Data Sources, Tables & Views

**Exam domain:** Using Spark SQL (20% of the exam)

**Time:** ~90 minutes

This is the domain where Synapse habits can hurt you: in Synapse you mostly write to ADLS paths with
`abfss://` and use `spark.sql` against Hive-metastore tables. The exam asks about the *generic* Spark
surface — save modes, `DataFrameReader`/`DataFrameWriter` options, querying files directly with SQL,
`saveAsTable` vs `save`, managed vs external tables, and the four kinds of view.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.appName("lab02-sql").master("local[4]").getOrCreate()

from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, IntegerType, StringType,
                               DoubleType, DateType, TimestampType)

# A writable scratch directory that works on Databricks and locally
try:
    BASE = "/Volumes/workspace/default/sparkcert"   # Free Edition: create this volume first
    dbutils.fs.mkdirs(BASE)                          # noqa: F821
except Exception:
    BASE = "/tmp/sparkcert"
    import shutil, os
    shutil.rmtree(BASE, ignore_errors=True); os.makedirs(BASE, exist_ok=True)
print("Writing to:", BASE)

In [ ]:
# Seed data
sales = spark.createDataFrame(
    [(1, "NL", "widget", 3, 9.99,  "2026-01-15"),
     (2, "BE", "widget", 1, 9.99,  "2026-01-15"),
     (3, "NL", "gadget", 5, 24.50, "2026-02-02"),
     (4, "DE", "gadget", 2, 24.50, "2026-02-11"),
     (5, "BE", "doodad", 7, 3.25,  "2026-03-01"),
     (6, "NL", "widget", 4, 9.99,  "2026-03-07")],
    "order_id INT, country STRING, product STRING, qty INT, price DOUBLE, order_date STRING"
).withColumn("order_date", F.to_date("order_date"))

sales.show()
sales.printSchema()

## 1. The DataFrameReader

```python
spark.read \
     .format("csv")            # csv | json | parquet | orc | delta | text | jdbc | avro
     .option("header", True)
     .schema(my_schema)        # or .option("inferSchema", True)
     .load("/path")            # or the shortcut: spark.read.csv("/path", header=True)
```

Things the exam likes:

- **`inferSchema` costs an extra pass over the data.** Supplying a schema is faster and safer. For JSON
  and CSV, inference is a *separate job*; for Parquet/ORC/Delta the schema is in the footer/log — free.
- **`mode` on read** (the *parse* mode, not the save mode!):
  `PERMISSIVE` (default — bad rows → nulls, raw row goes to `_corrupt_record`), `DROPMALFORMED`,
  `FAILFAST`.
- `spark.read.load(path)` with no format defaults to **Parquet** (`spark.sql.sources.default`).
- **Multiple paths**: `spark.read.parquet("/a", "/b")` or a list.
- **Schema as DDL string** is fully valid and much faster to type in an exam:
  `"order_id INT, country STRING"`.

In [ ]:
# Write some CSV so we have something to read badly
csv_path = f"{BASE}/sales_csv"
sales.write.mode("overwrite").option("header", True).csv(csv_path)

schema = StructType([
    StructField("order_id",   IntegerType(), True),
    StructField("country",    StringType(),  True),
    StructField("product",    StringType(),  True),
    StructField("qty",        IntegerType(), True),
    StructField("price",      DoubleType(),  True),
    StructField("order_date", DateType(),    True),
])

explicit = spark.read.format("csv").option("header", True).schema(schema).load(csv_path)
inferred = spark.read.format("csv").options(header=True, inferSchema=True).load(csv_path)

explicit.printSchema()
inferred.printSchema()    # compare: inference may give different types

In [ ]:
# Parse modes: PERMISSIVE vs FAILFAST vs DROPMALFORMED
bad = f"{BASE}/bad_csv"
spark.createDataFrame([("1","ok"), ("not_a_number","broken")], "a STRING, b STRING") \
     .write.mode("overwrite").option("header", True).csv(bad)

strict = StructType([StructField("a", IntegerType()), StructField("b", StringType())])

print("PERMISSIVE (default):")
spark.read.option("header", True).schema(strict).csv(bad).show()

print("DROPMALFORMED:")
spark.read.option("header", True).option("mode", "DROPMALFORMED").schema(strict).csv(bad).show()

print("FAILFAST would raise:")
try:
    spark.read.option("header", True).option("mode", "FAILFAST").schema(strict).csv(bad).show()
except Exception as e:
    print("  ->", type(e).__name__)

## 2. The DataFrameWriter and save modes

```python
df.write \
  .format("parquet") \
  .mode("overwrite")        # overwrite | append | ignore | error (a.k.a. errorifexists, the DEFAULT)
  .partitionBy("country") \
  .option("compression", "snappy") \
  .save("/path")            # -> files;  .saveAsTable("name") -> catalog table
```

| Mode | If the target exists |
|---|---|
| `error` / `errorifexists` | **throws** — this is the **default** |
| `overwrite` | replaces the data |
| `append` | adds to it |
| `ignore` | silently does nothing (a no-op, *not* an overwrite) |

`partitionBy` vs `bucketBy` vs `sortBy` — a guaranteed exam question:

- **`partitionBy(cols)`** → directory layout `country=NL/…`. Works with `save()` **and** `saveAsTable()`.
  Enables **partition pruning**. Use on low-cardinality columns.
- **`bucketBy(n, cols)`** → fixed number of files hashed by the column. **Only valid with
  `saveAsTable()`**, never with `save()`. Lets Spark skip the shuffle on later joins/aggregations on
  that column.
- **`sortBy(cols)`** → only valid **together with `bucketBy`**.

In [ ]:
parquet_path = f"{BASE}/sales_parquet"

(sales.write
      .mode("overwrite")
      .partitionBy("country")
      .parquet(parquet_path))

# Look at the directory layout that partitionBy produced
try:
    display(dbutils.fs.ls(parquet_path))     # noqa: F821  (Databricks)
except Exception:
    import os
    print(sorted(os.listdir(parquet_path)))

In [ ]:
# Partition pruning: the filter is pushed into the file listing, not applied after reading
pruned = spark.read.parquet(parquet_path).filter(F.col("country") == "NL")
pruned.explain("formatted")     # look for "PartitionFilters: [isnotnull(country), (country = NL)]"
pruned.show()

In [ ]:
# ignore vs append vs overwrite — prove the difference
tmp = f"{BASE}/mode_demo"
sales.write.mode("overwrite").parquet(tmp)
print("after overwrite:", spark.read.parquet(tmp).count())

sales.write.mode("ignore").parquet(tmp)      # no-op, data unchanged
print("after ignore   :", spark.read.parquet(tmp).count())

sales.write.mode("append").parquet(tmp)      # doubles the data
print("after append   :", spark.read.parquet(tmp).count())

## 3. Query files directly with SQL

Spark SQL can read a path with no table involved, using the ``format.`path` `` syntax
(**backticks**, not quotes — a classic exam trap):

```sql
SELECT * FROM parquet.`/path/to/data`
SELECT * FROM json.`/path/file.json`
SELECT * FROM csv.`/path/dir`      -- no header handling! every column is a string
SELECT * FROM delta.`/path/table`
SELECT * FROM text.`/path/file.txt`
SELECT * FROM binaryFile.`/path/*.png`
```

`csv.` and `text.` give you unparsed columns (`_c0`, `_c1`, … / `value`) because there is nowhere to put
options. For CSV with options, read it with the DataFrameReader and register a view instead.

In [ ]:
spark.sql(f"SELECT country, count(*) AS n FROM parquet.`{parquet_path}` GROUP BY country ORDER BY n DESC").show()
spark.sql(f"SELECT * FROM csv.`{csv_path}` LIMIT 3").show()   # note: header row appears as data

## 4. Views: temp, global temp, and (permanent) views

| Kind | Created with | Scope | Survives session? |
|---|---|---|---|
| Temporary view | `df.createOrReplaceTempView("v")` | this **SparkSession** | ❌ |
| Global temp view | `df.createOrReplaceGlobalTempView("v")` | all sessions in the **application**; must be referenced as `global_temp.v` | ❌ (dies with the app) |
| View (permanent) | `CREATE VIEW db.v AS SELECT …` | the catalog | ✅ |
| Table | `df.write.saveAsTable("t")` / `CREATE TABLE` | the catalog | ✅ |

Also: `createTempView` **throws** if the name exists; `createOrReplaceTempView` doesn't. Views store a
**query**, not data — nothing is materialised or cached by creating one.

In [ ]:
sales.createOrReplaceTempView("sales_v")
sales.filter("qty > 3").createOrReplaceGlobalTempView("big_orders")

spark.sql("""
    SELECT product,
           sum(qty)                AS units,
           round(sum(qty * price), 2) AS revenue
    FROM sales_v
    GROUP BY product
    ORDER BY revenue DESC
""").show()

spark.sql("SELECT * FROM global_temp.big_orders").show()
print("temp views:", [t.name for t in spark.catalog.listTables()])

## 5. Persistent tables — managed vs external

```python
df.write.saveAsTable("db.tbl")                     # MANAGED  — Spark owns data + metadata
df.write.option("path", "/some/path").saveAsTable("db.tbl")   # EXTERNAL — you own the data
```

- **DROP a managed table → the data is deleted.** DROP an external table → only the metadata goes.
  This is the single most-asked table question.
- `saveAsTable` respects `mode()`, `partitionBy()`, `bucketBy()`, `sortBy()`.
- `df.write.insertInto("tbl")` matches columns **by position**; `saveAsTable("tbl")` with `append`
  matches **by name**. Another favourite.
- On Databricks the default table format is **Delta**.

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS sparkcert")
spark.sql("USE sparkcert")

(sales.write
      .mode("overwrite")
      .partitionBy("country")
      .format("parquet")            # on Databricks you can leave this out to get Delta
      .saveAsTable("sales_managed"))

spark.sql("DESCRIBE EXTENDED sales_managed").show(50, truncate=False)

In [ ]:
# bucketBy: only legal with saveAsTable
(sales.write
      .mode("overwrite")
      .bucketBy(4, "product")
      .sortBy("order_id")
      .format("parquet")
      .saveAsTable("sales_bucketed"))

print("This raises, because bucketBy needs saveAsTable:")
try:
    sales.write.mode("overwrite").bucketBy(4, "product").parquet(f"{BASE}/nope")
except Exception as e:
    print("  ->", type(e).__name__, str(e)[:120])

## 6. JDBC

You will not have a database in the exam, but you must recognise the shape and the tuning options.

```python
(spark.read.format("jdbc")
      .option("url", "jdbc:postgresql://host:5432/db")
      .option("dbtable", "public.orders")      # or .option("query", "SELECT …")
      .option("user", "u").option("password", "p")
      .option("driver", "org.postgresql.Driver")
      # parallel reads — all four are required together:
      .option("partitionColumn", "order_id")
      .option("lowerBound", 1).option("upperBound", 1_000_000)
      .option("numPartitions", 8)
      .option("fetchsize", 10_000)
      .load())
```

Key facts:
- Without `partitionColumn`/`lowerBound`/`upperBound`/`numPartitions`, the read runs in **one task** on
  one executor — the whole table through a single connection.
- `partitionColumn` must be numeric, date or timestamp.
- `dbtable` and `query` are **mutually exclusive**.
- On write, `numPartitions` caps the number of concurrent connections; `batchsize` (default 1000) controls
  rows per round trip. `truncate=True` with `mode("overwrite")` truncates instead of dropping/recreating.

## 7. File formats — pick the right one

| Format | Splittable | Schema in file | Columnar | Predicate pushdown | Notes |
|---|---|---|---|---|---|
| **Parquet** | ✅ | ✅ | ✅ | ✅ | Spark's default; best general choice |
| **ORC** | ✅ | ✅ | ✅ | ✅ | similar to Parquet |
| **Delta** | ✅ | ✅ | ✅ | ✅ | Parquet + transaction log: ACID, time travel, schema enforcement |
| **Avro** | ✅ | ✅ | ❌ row-based | limited | good for streaming/row-wise writes |
| **JSON** | ✅ (line-delimited) | ❌ (inferred) | ❌ | ❌ | expensive to infer |
| **CSV** | ✅ | ❌ | ❌ | ❌ | no types, no nesting |

Compression matters: **snappy** (default for Parquet) and **lz4** are splittable-friendly;
**gzip** is **not splittable**, so one gzip file = one task no matter how big it is.

### ✍️ Exercises

**2.1** Write `sales` as JSON partitioned by `product`, then read it back with an explicit schema and count
the rows per product using pure SQL against the path (no view, no table).

**2.2** Create an external table over `parquet_path`, drop it, and prove the files are still there.

**2.3** What is the difference between these two, and when does the second one break?
```python
df.write.mode("append").saveAsTable("t")
df.write.insertInto("t")
```

<details><summary>Answers</summary>

**2.1**
```python
jp = f"{BASE}/sales_json"
sales.write.mode("overwrite").partitionBy("product").json(jp)
spark.sql(f"SELECT product, count(*) FROM json.`{jp}` GROUP BY product").show()
```
**2.2** `spark.sql(f"CREATE TABLE ext_sales USING parquet LOCATION '{parquet_path}'")`, then
`DROP TABLE ext_sales`, then `spark.read.parquet(parquet_path).count()` still works.

**2.3** `saveAsTable` in append mode resolves columns **by name**; `insertInto` resolves **by position**
and ignores the DataFrame's column names entirely. `insertInto` breaks (or silently corrupts data) when
your DataFrame's column order differs from the table's.
</details>

## 8. Wrap-up checklist

- [ ] I know the default save mode is `error`/`errorifexists`, and what `ignore` does.
- [ ] `bucketBy` needs `saveAsTable`; `sortBy` needs `bucketBy`.
- [ ] Dropping a **managed** table deletes the data; an external one doesn't.
- [ ] `SELECT * FROM parquet.` + **backticks** + path.
- [ ] Global temp views live in the `global_temp` database.
- [ ] `inferSchema` = extra pass; DDL-string schemas are valid.
- [ ] JDBC needs all four partitioning options together or it runs single-threaded.
- [ ] gzip is not splittable.